# Sesión 3: De Embeddings a Transformers

**Curso:** Introducción a Large Language Models  
**Profesor:** Francisco Pérez Carbajal  
**Basado en:** Hands-On Large Language Models + Stanford CS324  

---

## 📚 Contenido

1. **Sentence Embeddings**: Representar frases completas
2. **Similitud Semántica**: Comparar significados
3. **Búsqueda Semántica**: Encontrar contenido similar
4. **Visualización**: Ver embeddings en 2D
5. **Bonus**: Attention visualization (conceptual)

---

## Objetivos

- ✅ Usar sentence embeddings en código
- ✅ Calcular similitud entre frases
- ✅ Implementar búsqueda semántica básica
- ✅ Visualizar embeddings
- ✅ Entender cómo difieren de word embeddings

---

# Parte 1: Setup

## Instalar Dependencias

In [ ]:
# Instalar sentence-transformers
!pip install sentence-transformers scikit-learn matplotlib seaborn plotly -q

In [ ]:
# Imports
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Imports completados")

---

# Parte 2: Sentence Embeddings

## ¿Qué son Sentence Embeddings?

A diferencia de word2vec que representa **palabras individuales**, 
sentence embeddings representan **frases completas** como vectores.

```
Word Embeddings (Word2Vec):
"gato" → [0.2, 0.8, -0.3, ...]  (300D)

Sentence Embeddings:
"El gato come pescado" → [0.1, 0.4, 0.7, ...]  (384D o 768D)
```

**Ventaja:** Captura el significado de la frase COMPLETA

## Cargar Modelo Pre-entrenado

In [ ]:
# Cargar modelo de sentence-transformers
# Este es un modelo pequeño y rápido, perfecto para empezar

print("Cargando modelo...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✓ Modelo cargado")
print(f"✓ Dimensión de embeddings: {model.get_sentence_embedding_dimension()}D")
print(f"✓ Max sequence length: {model.max_seq_length} tokens")

## Crear Embeddings de Oraciones

In [ ]:
# Oraciones de ejemplo
oraciones = [
    "El gato come pescado",
    "El perro come carne",
    "El clima está soleado",
    "Un felino se alimenta de peces",  # Similar a la primera
    "Hace calor hoy"  # Similar a la tercera
]

# Generar embeddings
embeddings = model.encode(oraciones)

print(f"Número de oraciones: {len(oraciones)}")
print(f"Shape de embeddings: {embeddings.shape}")
print(f"\nPrimera oración: '{oraciones[0]}'")
print(f"Embedding (primeros 10 valores): {embeddings[0][:10]}")

---

# Parte 3: Similitud Semántica

## Calcular Similitud Entre Oraciones

In [ ]:
# Calcular matriz de similitud
similitud_matrix = cosine_similarity(embeddings)

# Crear DataFrame para mejor visualización
df_similitud = pd.DataFrame(
    similitud_matrix,
    index=[f"S{i+1}" for i in range(len(oraciones))],
    columns=[f"S{i+1}" for i in range(len(oraciones))]
)

print("Matriz de Similitud (cosine similarity):\n")
print(df_similitud.round(3))
print("\nLeyenda de oraciones:")
for i, oracion in enumerate(oraciones, 1):
    print(f"S{i}: {oracion}")

## Visualizar Similitud con Heatmap

In [ ]:
# Crear heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    similitud_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    xticklabels=[f"S{i+1}" for i in range(len(oraciones))],
    yticklabels=[f"S{i+1}" for i in range(len(oraciones))],
    vmin=0,
    vmax=1,
    square=True
)

plt.title('Matriz de Similitud Semántica', fontsize=16, fontweight='bold')
plt.xlabel('Oraciones', fontsize=12)
plt.ylabel('Oraciones', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 Observaciones:")
print("  • Diagonal = 1.0 (cada oración consigo misma)")
print("  • Verde oscuro = Alta similitud")
print("  • Rojo = Baja similitud")
print("\n  ¿Cuáles son las oraciones más similares?")

## Encontrar Oraciones Más Similares

In [ ]:
# Función para encontrar oraciones más similares
def encontrar_similares(query_idx, top_k=3):
    """
    Encuentra las oraciones más similares a una query
    """
    similitudes = similitud_matrix[query_idx]
    
    # Obtener índices ordenados (excluyendo la query misma)
    indices_ordenados = np.argsort(similitudes)[::-1][1:top_k+1]
    
    print(f"\n🔍 Query: '{oraciones[query_idx]}'\n")
    print("Oraciones más similares:")
    print("-" * 60)
    
    for i, idx in enumerate(indices_ordenados, 1):
        sim = similitudes[idx]
        print(f"{i}. [{sim:.3f}] {oraciones[idx]}")

# Probar con la primera oración
encontrar_similares(0)

# Probar con la tercera oración
encontrar_similares(2)

---

# Parte 4: Tu Turno - Experimenta

## Agrega Tus Propias Oraciones

In [ ]:
# ¡Agrega tus propias oraciones aquí!
mis_oraciones = [
    "Python es un lenguaje de programación",
    "Los perros son animales leales",
    "Machine learning es una rama de la inteligencia artificial",
    "Los gatos son mascotas independientes",
    "Deep learning utiliza redes neuronales"
]

# Generar embeddings
mis_embeddings = model.encode(mis_oraciones)

# Calcular similitudes
mis_similitudes = cosine_similarity(mis_embeddings)

# Visualizar
plt.figure(figsize=(12, 10))
sns.heatmap(
    mis_similitudes,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    xticklabels=[f"S{i+1}" for i in range(len(mis_oraciones))],
    yticklabels=[f"S{i+1}" for i in range(len(mis_oraciones))],
    square=True
)

plt.title('Mis Oraciones - Similitud Semántica', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nLeyenda:")
for i, oracion in enumerate(mis_oraciones, 1):
    print(f"S{i}: {oracion}")

---

# Parte 5: Búsqueda Semántica

## Implementar Motor de Búsqueda Simple

In [ ]:
# Base de conocimiento (documentos)
documentos = [
    "Los transformers usan mecanismo de attention para procesar secuencias",
    "BERT es un modelo de lenguaje pre-entrenado basado en transformers",
    "GPT-3 tiene 175 mil millones de parámetros",
    "Word2vec crea embeddings de palabras usando redes neuronales",
    "Los embeddings capturan similitud semántica entre palabras",
    "Attention permite que los modelos se enfoquen en partes relevantes",
    "Fine-tuning adapta modelos pre-entrenados a tareas específicas",
    "Los LLMs pueden generar texto coherente y relevante",
    "Self-attention calcula relaciones dentro de una secuencia",
    "Los embeddings de oraciones capturan el significado completo"
]

# Crear índice (embeddings de todos los documentos)
print("Creando índice de búsqueda...")
doc_embeddings = model.encode(documentos, show_progress_bar=True)
print(f"✓ Índice creado con {len(documentos)} documentos")

In [ ]:
# Función de búsqueda semántica
def buscar(query, top_k=3):
    """
    Búsqueda semántica en la base de documentos
    """
    # Crear embedding de la query
    query_embedding = model.encode([query])
    
    # Calcular similitudes
    similitudes = cosine_similarity(query_embedding, doc_embeddings)[0]
    
    # Obtener top_k documentos
    top_indices = np.argsort(similitudes)[::-1][:top_k]
    
    print(f"\n🔍 Búsqueda: '{query}'\n")
    print("Resultados más relevantes:")
    print("=" * 70)
    
    for i, idx in enumerate(top_indices, 1):
        score = similitudes[idx]
        print(f"\n{i}. [Score: {score:.3f}]")
        print(f"   {documentos[idx]}")
    
    return top_indices

# Probar búsquedas
buscar("¿Qué es attention?")

In [ ]:
# Más búsquedas de ejemplo
buscar("embeddings de palabras")

In [ ]:
buscar("modelos grandes de lenguaje")

### 🎯 Tu Turno: Prueba Tus Búsquedas

In [ ]:
# Prueba con tu propia query
mi_busqueda = ""  # Escribe tu query aquí

if mi_busqueda:
    buscar(mi_busqueda)
else:
    print("Escribe una query en 'mi_busqueda' y ejecuta de nuevo")

---

# Parte 6: Visualización en 2D

## Reducir Dimensionalidad y Graficar

In [ ]:
# Preparar datos para visualización
todas_oraciones = [
    "El gato come pescado",
    "El perro come carne",
    "El felino se alimenta",
    "Python es un lenguaje",
    "Machine learning es IA",
    "Deep learning usa redes",
    "Hace calor hoy",
    "El clima está soleado",
    "Llueve mucho"
]

# Etiquetas por categoría
categorias = [
    'Animales', 'Animales', 'Animales',
    'Programación', 'Programación', 'Programación',
    'Clima', 'Clima', 'Clima'
]

# Generar embeddings
print("Generando embeddings...")
embeddings_viz = model.encode(todas_oraciones)
print(f"Shape original: {embeddings_viz.shape}")

# Reducir a 2D con t-SNE
print("Reduciendo dimensiones con t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
embeddings_2d = tsne.fit_transform(embeddings_viz)
print(f"Shape reducido: {embeddings_2d.shape}")
print("✓ Listo para visualizar")

In [ ]:
# Crear visualización interactiva
plt.figure(figsize=(14, 10))

# Colores por categoría
colores = {'Animales': 'red', 'Programación': 'blue', 'Clima': 'green'}

for categoria in set(categorias):
    # Filtrar puntos de esta categoría
    indices = [i for i, c in enumerate(categorias) if c == categoria]
    x = embeddings_2d[indices, 0]
    y = embeddings_2d[indices, 1]
    
    # Graficar
    plt.scatter(x, y, c=colores[categoria], label=categoria, s=200, alpha=0.6, edgecolors='black')

# Añadir etiquetas
for i, texto in enumerate(todas_oraciones):
    # Truncar texto si es muy largo
    texto_corto = texto[:30] + '...' if len(texto) > 30 else texto
    plt.annotate(
        texto_corto,
        xy=(embeddings_2d[i, 0], embeddings_2d[i, 1]),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.3)
    )

plt.title('Visualización de Sentence Embeddings en 2D', fontsize=16, fontweight='bold')
plt.xlabel('Dimensión 1 (t-SNE)', fontsize=12)
plt.ylabel('Dimensión 2 (t-SNE)', fontsize=12)
plt.legend(fontsize=12, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Observaciones:")
print("  • Oraciones similares están cerca en el espacio")
print("  • Cada color = una categoría temática")
print("  • Clusters visibles por tema")

---

# Parte 7: Comparación con Word Embeddings

## Diferencia Clave

In [ ]:
# Ejemplo: La palabra "bank" en diferentes contextos
oraciones_bank = [
    "I went to the bank to deposit money",
    "I sat by the river bank",
    "The bank offers great interest rates",
    "The bank of the river was muddy"
]

# Generar sentence embeddings
embeddings_bank = model.encode(oraciones_bank)

# Calcular similitudes
sim_matrix = cosine_similarity(embeddings_bank)

# Visualizar
plt.figure(figsize=(10, 8))
sns.heatmap(
    sim_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    xticklabels=[f"S{i+1}" for i in range(len(oraciones_bank))],
    yticklabels=[f"S{i+1}" for i in range(len(oraciones_bank))],
    square=True
)

plt.title('Contexto de "bank" - Sentence Embeddings', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nOraciones:")
for i, oracion in enumerate(oraciones_bank, 1):
    print(f"S{i}: {oracion}")

print("\n💡 Observa:")
print("  • S1 y S3 (banco financiero) son más similares entre sí")
print("  • S2 y S4 (orilla del río) son más similares entre sí")
print("  • ¡Sentence embeddings CAPTURAN el contexto!")
print("\n  Con word2vec, 'bank' sería SIEMPRE el mismo vector")

---

# Parte 8: Bonus - Visualización de Attention (Conceptual)

## Simulación Simple de Attention Weights

In [ ]:
# Simular attention weights para demostración
# (En realidad esto vendría del modelo, aquí lo simulamos)

input_words = ["I", "love", "llamas"]
output_words = ["Ik", "hou", "van", "lama's"]

# Matriz de attention simulada (valores de ejemplo)
attention_weights = np.array([
    [0.8, 0.1, 0.1],  # "Ik" atiende más a "I"
    [0.1, 0.8, 0.1],  # "hou" atiende más a "love"
    [0.1, 0.7, 0.2],  # "van" atiende más a "love"
    [0.1, 0.1, 0.8]   # "lama's" atiende más a "llamas"
])

# Visualizar
plt.figure(figsize=(10, 8))
sns.heatmap(
    attention_weights,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    xticklabels=input_words,
    yticklabels=output_words,
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Attention Weight'}
)

plt.title('Ejemplo de Attention Weights\n(Traducción: "I love llamas" → "Ik hou van lama\'s")', 
          fontsize=14, fontweight='bold')
plt.xlabel('Input (English)', fontsize=12)
plt.ylabel('Output (Dutch)', fontsize=12)
plt.tight_layout()
plt.show()

print("\n💡 Interpretación:")
print("  • Colores cálidos (amarillo-rojo) = Alta atención")
print("  • Colores fríos (blanco) = Baja atención")
print("  • Cada fila muestra a qué palabras del input atiende cada palabra del output")
print("\n  Ejemplo: 'lama\\'s' atiende principalmente a 'llamas' (0.8)")

---

# Resumen y Reflexión

## ✅ Lo Que Hicimos Hoy

### 1. Sentence Embeddings
- ✓ Representar frases completas como vectores
- ✓ A diferencia de word2vec (palabras individuales)
- ✓ Capturan significado completo de la oración

### 2. Similitud Semántica
- ✓ Calcular qué tan similares son dos frases
- ✓ Cosine similarity como métrica
- ✓ Visualización con heatmaps

### 3. Búsqueda Semántica
- ✓ Motor de búsqueda simple
- ✓ Encuentra documentos relevantes
- ✓ Basado en significado, no keywords

### 4. Visualización
- ✓ Reducción a 2D con t-SNE
- ✓ Clusters visibles por tema
- ✓ Oraciones similares cerca en espacio

### 5. Contexto
- ✓ Sentence embeddings capturan contexto
- ✓ "Bank" tiene diferentes significados según contexto
- ✓ Ventaja sobre word embeddings estáticos

---

## 💡 Insights Clave

1. **Progresión:**
   ```
   Word Embeddings → Sentence Embeddings → (próximo: Transformers)
   ```

2. **Aplicaciones Prácticas:**
   - Búsqueda semántica (Google, ChatGPT)
   - Sistemas de recomendación
   - Detección de duplicados
   - Clasificación de texto

3. **Conexión con Transformers:**
   - Modelos como BERT crean embeddings contextuales
   - Attention permite capturar relaciones
   - Base para LLMs modernos

---

## 🔮 Próximos Pasos

**En clase:**
- Entender arquitectura Transformer
- Cómo funciona attention en detalle
- Capabilities de GPT-3

**Para explorar (opcional):**
- Probar otros modelos de sentence-transformers
- Implementar tu propia búsqueda semántica
- Experimentar con diferentes idiomas

---

## 📚 Recursos

- **Sentence-Transformers:** https://www.sbert.net/
- **Hugging Face:** https://huggingface.co/sentence-transformers
- **Documentación:** https://www.sbert.net/docs/pretrained_models.html
- **Stanford CS324:** https://stanford-cs324.github.io/
- **Paper Attention:** Bahdanau et al. (2014)
- **Paper Transformers:** Vaswani et al. (2017) "Attention is All You Need"
- **Paper GPT-3:** Brown et al. (2020)
- **Visualización:** https://jalammar.github.io/illustrated-transformer/


**¿Dudas?** franciscop@ciencias.unam.mx